# 09 — Local DINOv2 model restore (Five-role 2.5D, one fold)

Notebook này tiếp nối `08_local_dinov2_preprocessing.ipynb`.

**Mục tiêu Phase 2B:** dựng đúng kiến trúc `FiveRoleDINOv2_2p5D` từ notebook Kaggle ensemble cuối và load **một** checkpoint fold bằng `strict=True`.

Nguồn kiến trúc:
- `08_model_ensemble_kaggle_dataset_ready.ipynb`, Cell 29 — exact model class.
- Cùng notebook, Cell 31 — local DINOv2 backbone construction và checkpoint metadata checks.
- `bundle_manifest.json` — image/patch/backbone/embed dimensions, target order, role order và fold metadata.

**Chưa load cả 5 folds, chưa làm five-fold averaging, chưa blend với ViT.** Chỉ load fold 1 để xác thực compatibility. Không training và không thay đổi notebook competition.

## Input cần có

1. Phase 2A output: `demo/output/dinov2_preprocessing/dinov2_preprocessed_sample.npz`.
2. **Final five-fold DINOv2 bundle** có đủ:
   - `bundle_manifest.json`
   - `dinov2_repo/hubconf.py` và `dinov2_repo/dinov2/`
   - `five_role_2p5d_v2_cv5/fold_1/best_checkpoint.pt` đến `fold_5/best_checkpoint.pt`

> Không dùng `artifacts/teacher/dinov2-artifact.zip` (artifact cũ, 8 MB trong danh sách trước đó) trừ khi nó thật sự chứa đủ manifest, local source và 5 checkpoint cuối. Loader sẽ từ chối bundle thiếu thành phần.

Notebook tự dùng CUDA nếu khả dụng, nếu không sẽ chạy CPU. Trên CPU, encoder chunk mặc định là 2 để giảm memory; trên CUDA mặc định là 8 như notebook Kaggle.


In [1]:
# 1. Project paths, imports and runtime
from pathlib import Path
import sys
import os
import json
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from rsna_knee.models import (
    TARGETS,
    FIVE_ROLE_ORDER,
    find_bundle_root,
    resolve_bundle_root,
    validate_bundle_root,
    load_dinov2_fold,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER_CHUNK_SIZE = 8 if DEVICE.type == "cuda" else 2

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Encoder chunk size:", ENCODER_CHUNK_SIZE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available: model restore will use CPU.")


Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Python: 3.13.9
PyTorch: 2.11.0+cpu
Device: cpu
Encoder chunk size: 2
CUDA not available: model restore will use CPU.


## 2. Locate the Phase 2A preprocessed sample

The NPZ file contains the exact Phase 2A model-input arrays. It is not a label file or prediction output.

In [2]:
PREPROCESSED_PATH = (
    PROJECT_ROOT / "demo" / "output" / "dinov2_preprocessing"
    / "dinov2_preprocessed_sample.npz"
)

if not PREPROCESSED_PATH.is_file():
    raise FileNotFoundError(
        f"Phase 2A NPZ not found: {PREPROCESSED_PATH}\n"
        "Run notebooks/08_local_dinov2_preprocessing.ipynb first, or update PREPROCESSED_PATH."
    )

with np.load(PREPROCESSED_PATH, allow_pickle=False) as data:
    images_np = data["images"].copy()
    window_valid_np = data["window_valid"].copy()
    window_positions_np = data["window_positions"].copy()
    saved_role_order = data["role_order"].astype(str).tolist()
    role_series_uids = data["role_series_uids"].astype(str).tolist()

assert saved_role_order == list(FIVE_ROLE_ORDER), (
    f"Role order mismatch: {saved_role_order} != {list(FIVE_ROLE_ORDER)}"
)
assert images_np.shape == (5, 10, 3, 224, 224), images_np.shape
assert images_np.dtype == np.uint8, images_np.dtype
assert window_valid_np.shape == (5, 10), window_valid_np.shape
assert window_positions_np.shape == (5, 10), window_positions_np.shape
assert np.isfinite(window_positions_np).all()

print("Preprocessed sample:", PREPROCESSED_PATH)
print("images:", images_np.shape, images_np.dtype)
print("window_valid:", window_valid_np.shape, window_valid_np.dtype)
print("window_positions:", window_positions_np.shape, window_positions_np.dtype)
print("Valid windows:", int(window_valid_np.sum()), "/", int(window_valid_np.size))
print("Role availability:", window_valid_np.astype(bool).any(axis=1).tolist())
for role, series_uid in zip(FIVE_ROLE_ORDER, role_series_uids):
    print(f"  {role}: {series_uid}")


Preprocessed sample: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_preprocessing\dinov2_preprocessed_sample.npz
images: (5, 10, 3, 224, 224) uint8
window_valid: (5, 10) uint8
window_positions: (5, 10) float32
Valid windows: 50 / 50
Role availability: [True, True, True, True, True]
  SAG_FLUID: 1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429
  COR_FLUID: 1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198
  AX_FLUID: 1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430
  SAG_NONFLUID: 1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282
  COR_NONFLUID: 1.2.826.0.1.3680043.8.498.12451507871565834868863189854063726441


## 3. Locate the final five-fold DINOv2 bundle

Set `DINO_BUNDLE_SOURCE` to either:
- the extracted bundle directory containing `bundle_manifest.json`, or
- the full `five_role_v2_cv5_submission_bundle.zip` downloaded from Kaggle.

If left blank, the notebook searches under `artifacts/` for one complete extracted bundle. It does **not** select an older generic DINO artifact merely by filename.

In [7]:
# IMPORTANT: set this if the bundle is outside project_root/artifacts.
# Examples:
# DINO_BUNDLE_SOURCE = PROJECT_ROOT / "artifacts" / "dinov2" / "five_role_v2_cv5_submission_bundle"
DINO_BUNDLE_SOURCE = PROJECT_ROOT / "artifacts" / "five_role_v2_cv5_submission_bundle.zip"
DINO_BUNDLE_SOURCE = None

if DINO_BUNDLE_SOURCE is None:
    artifact_root = PROJECT_ROOT / "artifacts"
    if not artifact_root.exists():
        raise FileNotFoundError(
            f"Artifact folder not found: {artifact_root}. Set DINO_BUNDLE_SOURCE explicitly."
        )
    try:
        bundle_root = find_bundle_root(artifact_root)
    except Exception as directory_error:
        # Also detect the explicitly named final 5-fold ZIP, but never guess
        # that a generic/older dinov2-artifact.zip is the final bundle.
        zip_candidates = list(
            artifact_root.rglob("*five*role*cv5*submission*bundle*.zip")
        )
        if len(zip_candidates) == 1:
            bundle_cache = artifact_root / "_extracted_final_dinov2_bundle"
            bundle_root = resolve_bundle_root(zip_candidates[0], bundle_cache)
        else:
            raise RuntimeError(
                "Could not auto-discover a complete final five-fold DINOv2 bundle under artifacts/.\n"
                "Set DINO_BUNDLE_SOURCE to the extracted final bundle directory or its full ZIP.\n"
                "The small artifacts/teacher/dinov2-artifact.zip is not assumed to be the final bundle."
            ) from directory_error
else:
    bundle_source = Path(DINO_BUNDLE_SOURCE)
    bundle_cache = PROJECT_ROOT / "artifacts" / "_extracted_final_dinov2_bundle"
    bundle_root = resolve_bundle_root(
        bundle_source,
        bundle_cache if bundle_source.suffix.lower() == ".zip" else None,
    )

bundle_root, bundle_manifest = validate_bundle_root(bundle_root)
print("Resolved DINOv2 bundle:", bundle_root)
print("Architecture:", bundle_manifest["architecture"])
print("Backbone:", bundle_manifest["backbone_model"])
print("Image / patch size:", bundle_manifest["image_size"], "/", bundle_manifest["patch_size"])
print("Backbone / embed dim:", bundle_manifest["backbone_dim"], "/", bundle_manifest["embed_dim"])
print("Freeze early blocks:", bundle_manifest["freeze_early_blocks"])
print("Role order:", bundle_manifest["role_order"])
print("Targets:", bundle_manifest["targets"])
print("Fold checkpoint files:")
for fold in range(1, 6):
    checkpoint_path = (
        bundle_root / "five_role_2p5d_v2_cv5" / f"fold_{fold}" / "best_checkpoint.pt"
    )
    print(f"  fold {fold}: exists={checkpoint_path.is_file()} | size={checkpoint_path.stat().st_size / 1024**2:.1f} MiB")


Resolved DINOv2 bundle: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle
Architecture: DINOv2 ViT-S/14 shared 2.5D five-role; target-by-role attention; 12 independent heads
Backbone: dinov2_vits14
Image / patch size: 224 / 14
Backbone / embed dim: 384 / 256
Freeze early blocks: 9
Role order: ['SAG_FLUID', 'COR_FLUID', 'AX_FLUID', 'SAG_NONFLUID', 'COR_NONFLUID']
Targets: ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
Fold checkpoint files:
  fold 1: exists=True | size=93.0 MiB
  fold 2: exists=True | size=93.0 MiB
  fold 3: exists=True | size=93.0 MiB
  fold 4: exists=True | size=93.0 MiB
  fold 5: exists=True | size=93.0 MiB


In [ ]:
# import zipfile
# import pandas as pd

# # Thay đổi đường dẫn đến file ZIP của bạn
# zip_path = 'C:/Users/THINH DINH/Desktop/space/rsna-knee-abnormality-detection/artifacts/teacher/dinov2-artifact.zip'

# with zipfile.ZipFile(zip_path, 'r') as z:
#     data = [
#         {
#             'Tên file': item.filename,
#             'Kích thước gốc (KB)': round(item.file_size / 1024, 2),
#             'Kích thước nén (KB)': round(item.compress_size / 1024, 2),
#             'Thời gian sửa đổi': f"{item.date_time[0]}-{item.date_time[1]:02d}-{item.date_time[2]:02d} {item.date_time[3]:02d}:{item.date_time[4]:02d}"
#         }
#         for item in z.infolist()
#     ]

# # Hiển thị dữ liệu dạng bảng đẹp mắt trên Jupyter Notebook
# df = pd.DataFrame(data)
# df

,Tên file,Kích thước gốc (KB),Kích thước nén (KB),Thời gian sửa đổi
0,cnn_lstm_best.pt,7721.21,7104.62,2026-10-16 05:27
1,cnn_lstm_gold_validation_metrics.csv,0.72,0.28,2026-10-16 05:29
2,cnn_lstm_gold_validation_predictions.csv,13.80,2.42,2026-10-16 05:30
3,cnn_lstm_manifest.json,1.45,0.74,2026-10-16 05:30
4,cnn_lstm_training_log.csv,0.50,0.28,2026-10-16 05:30
5,preprocess_status.csv,481.01,130.02,2026-10-16 05:30
6,selected_series_index.csv,1854.99,397.93,2026-10-16 05:30
7,shared_split.csv,336.05,87.71,2026-10-16 05:31
8,study_labels_training.csv,493.89,113.98,2026-10-16 05:32


## 4. Restore one fold and enforce strict checkpoint compatibility

This cell instantiates the backbone from the local bundled DINOv2 source with `pretrained=False`, then loads the full fold checkpoint (including the fine-tuned backbone weights). `strict=True` must succeed with no missing or unexpected state-dict keys.

In [8]:
FOLD_TO_TEST = 1

model, model_info = load_dinov2_fold(
    bundle_root=bundle_root,
    fold=FOLD_TO_TEST,
    device=DEVICE,
    encoder_chunk_size=ENCODER_CHUNK_SIZE,
)

print("========== DINOv2 FOLD RESTORE ==========")
print("Fold:", model_info["fold"])
print("Best epoch:", model_info["best_epoch"])
print("Best validation loss:", model_info["best_val_loss"])
print("Strict state-dict load:", model_info["strict_load"])
print("Parameters (M):", round(model_info["parameter_count"] / 1e6, 3))
print("Trainable parameters (M):", round(model_info["trainable_parameter_count"] / 1e6, 3))
print("Device:", model_info["device"])
print("Checkpoint:", model_info["checkpoint_path"])
assert model_info["strict_load"] is True


C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


========== DINOv2 FOLD RESTORE ==========
Fold: 1
Best epoch: 2
Best validation loss: 0.6435930132865906
Strict state-dict load: True
Parameters (M): 24.349
Trainable parameters (M): 7.619
Device: cpu
Checkpoint: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\five_role_2p5d_v2_cv5\fold_1\best_checkpoint.pt


## 5. Quick forward smoke test (one valid window per role)

The quick test keeps the full expected input shape `[1, 5, 10, 3, 224, 224]`, but marks only one valid window in each role. The model therefore encodes five windows instead of all fifty. This tests the real DINOv2 source, restored weights, tensor contract and 12-logit output without performing the full study inference.

This is a **software smoke test**, not a medical result, validation metric, or competition score.

In [9]:
RUN_QUICK_FORWARD_TEST = True

if RUN_QUICK_FORWARD_TEST:
    valid_source = window_valid_np.astype(bool)
    quick_valid = np.zeros_like(valid_source, dtype=np.bool_)
    quick_positions = np.zeros_like(window_positions_np, dtype=np.float32)

    for role_idx in range(len(FIVE_ROLE_ORDER)):
        indices = np.flatnonzero(valid_source[role_idx])
        if len(indices) == 0:
            raise RuntimeError(f"No valid windows available for role {FIVE_ROLE_ORDER[role_idx]}.")
        selected_idx = int(indices[len(indices) // 2])
        quick_valid[role_idx, selected_idx] = True
        quick_positions[role_idx, selected_idx] = window_positions_np[role_idx, selected_idx]

    images_t = torch.from_numpy(images_np[None]).to(DEVICE)
    valid_t = torch.from_numpy(quick_valid[None]).to(DEVICE)
    positions_t = torch.from_numpy(quick_positions[None]).to(DEVICE)

    with torch.inference_mode():
        logits = model(images_t, valid_t, positions_t)
        probabilities = torch.sigmoid(logits.float()).squeeze(0).cpu().numpy()

    assert logits.shape == (1, len(TARGETS)), tuple(logits.shape)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0.0) & (probabilities <= 1.0)).all()

    print("Quick forward test: PASS")
    print("Windows encoded:", int(quick_valid.sum()), "/ 50")
    print("Logits shape:", tuple(logits.shape))
    print("Probabilities finite and in [0, 1]: PASS")
else:
    print("Quick forward test skipped by user configuration.")


Quick forward test: PASS
Windows encoded: 5 / 50
Logits shape: (1, 12)
Probabilities finite and in [0, 1]: PASS


## 6. Save restore audit and completion criteria

In [10]:
from datetime import datetime, timezone

OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "dinov2_model_restore"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

restore_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook": "08_model_ensemble_kaggle_dataset_ready.ipynb",
    "source_cells": {"architecture": 29, "checkpoint_restore": 31},
    "bundle_root": str(bundle_root),
    "bundle_architecture": bundle_manifest.get("architecture"),
    "study_input": str(PREPROCESSED_PATH),
    "fold_tested": FOLD_TO_TEST,
    "model_info": model_info,
    "preprocessed_contract": {
        "images_shape": list(images_np.shape),
        "images_dtype": str(images_np.dtype),
        "window_valid_shape": list(window_valid_np.shape),
        "window_positions_shape": list(window_positions_np.shape),
        "role_order": list(saved_role_order),
        "valid_windows": int(window_valid_np.sum()),
    },
    "quick_forward_test": {
        "ran": bool(RUN_QUICK_FORWARD_TEST),
        "valid_windows_encoded": int(quick_valid.sum()) if RUN_QUICK_FORWARD_TEST else 0,
        "logits_shape": list(logits.shape) if RUN_QUICK_FORWARD_TEST else None,
        "probabilities_finite": bool(np.isfinite(probabilities).all()) if RUN_QUICK_FORWARD_TEST else None,
    },
    "note": "One-fold software restore/smoke test only. No five-fold ensemble, ViT blend, or performance evaluation was performed.",
}
report_path = OUTPUT_DIR / "dinov2_fold1_restore_report.json"
report_path.write_text(json.dumps(restore_report, indent=2), encoding="utf-8")
print("Saved restore report:", report_path)


Saved restore report: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_model_restore\dinov2_fold1_restore_report.json


### Phase 2B completion criteria

- [ ] The notebook resolves the **final** bundle (manifest + local DINOv2 source + all five best checkpoints).
- [ ] Bundle manifest target order and role order exactly match the Kaggle model.
- [ ] Fold 1 metadata says `fold == 1` and contains a non-empty `model_state`.
- [ ] `load_state_dict(..., strict=True)` succeeds with no missing/unexpected keys.
- [ ] Model is in `eval()` mode and placed on the selected local device.
- [ ] Quick forward smoke test returns finite logits with shape `[1, 12]`.
- [ ] `dinov2_fold1_restore_report.json` is saved.

**Phase 2B scope boundary:** it does not yet run all fifty windows for one study, load all five folds simultaneously, average fold probabilities, or blend with ViT. Those belong to the next inference phase.